# 🚀 TopAI — Free AI Playground

### Run AI models in the cloud — locally in your notebook, or through free APIs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghostdev102/TopAI/blob/main/ai.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ghostdev102/TopAI/main?labpath=ai.ipynb)
[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ghostdev102/TopAI/blob/main/ai.ipynb)

---

## 🧠 What is TopAI?

**TopAI** is a free-AI playground for experimenting with modern AI models without locking yourself into one provider.

This notebook supports two completely different modes:

| Mode | Inference | API key | Root |
|---|---|---:|---:|
| 🖥️ Local | Your notebook runtime | ❌ | ❌ |
| ☁️ API | Cloud provider | Usually required | ❌ |

> **No root. No `sudo`. No mandatory paid API.**

The local mode downloads an open model and performs inference directly inside the current notebook runtime.

---

## 🏆 Notebook Features

- ⚡ Automatic CPU/GPU detection
- 🧠 Hugging Face model support
- 💬 Interactive chat UI
- 🎛️ Generation controls
- 📊 Hardware inspector
- 🚀 Tokens/second benchmark
- 💾 Hugging Face cache support
- 🔐 No API key for local mode
- ☁️ OpenAI-compatible API support
- 📓 Works with Jupyter-style environments
- 🧑‍💻 Designed for normal users — no root required

### The cells are deliberately modular.

**Cell 1:** Bootstrap the environment  
**Cell 2:** Inspect your hardware  
**Cell 3:** Select your model  
**Cell 4:** Download and load the model  
**Cell 5:** Build the inference engine  
**Cell 6:** Run your first prompt  
**Cell 7:** Benchmark the runtime  
**Cell 8:** Launch the interactive chat UI  
**Cell 9:** Optional cloud API mode  
**Cell 10:** TopAI scorecard


## ⚠️ Important

Hosted notebook environments have limited CPU, RAM, disk, and sometimes GPU resources.

The default model is intentionally small so the notebook is practical on common hosted environments.

Larger models require more memory. A notebook cannot bypass the hardware limits of the runtime it is running on.

In [ ]:
# ============================================================
# 🚀 CELL 1 — TOPAI BOOTSTRAP
# ============================================================

# Everything is installed for the current user.
# No sudo and no root privileges are required.

%pip install --user -q -U transformers accelerate sentencepiece huggingface_hub psutil ipywidgets

import sys

print("╔══════════════════════════════════════════════╗")
print("║              🚀 TOPAI BOOTSTRAP             ║")
print("╚══════════════════════════════════════════════╝")
print()
print("🐍 Python:", sys.version.split()[0])
print("✅ User-space environment ready")


In [ ]:
# ============================================================
# 🔥 CELL 2 — HARDWARE INSPECTOR
# ============================================================

import os
import platform
import psutil
import torch

RAM_GB = psutil.virtual_memory().total / 1024**3
GPU_AVAILABLE = torch.cuda.is_available()

print("╔══════════════════════════════════════════════╗")
print("║           🔥 TOPAI HARDWARE CHECK           ║")
print("╚══════════════════════════════════════════════╝")
print()
print("🖥️  OS       :", platform.system())
print("🐍 Python   :", platform.python_version())
print("🧠 CPU      :", os.cpu_count(), "threads")
print(f"💾 RAM      : {RAM_GB:.2f} GB")
print("🔥 PyTorch  :", torch.__version__)
print("🎮 GPU      :", GPU_AVAILABLE)

if GPU_AVAILABLE:
    print("🚀 GPU NAME :", torch.cuda.get_device_name(0))
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 VRAM     : {VRAM_GB:.2f} GB")
else:
    print("🐢 Mode     : CPU inference")

print()
print("✅ Hardware inspection complete")


# 🧠 Model Selection

Pick an open model from Hugging Face.

The default is intentionally lightweight.

### Suggested progression

| Model | Approximate class | Hosted-runtime friendliness |
|---|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` | Small | 🟢 Excellent |
| `Qwen/Qwen2.5-1.5B-Instruct` | Small | 🟢 Good |
| `Qwen/Qwen2.5-3B-Instruct` | Medium | 🟡 Depends on RAM |

Larger models may require a GPU or substantially more RAM.

In [ ]:
# ============================================================
# 🧠 CELL 3 — MODEL SELECTOR
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

AVAILABLE_MODELS = {
    "Small — Qwen 0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "Small — Qwen 1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Medium — Qwen 3B": "Qwen/Qwen2.5-3B-Instruct",
}

print("🎯 Selected model:")
print("   ", MODEL_NAME)
print()
print("Available examples:")
for name, model_id in AVAILABLE_MODELS.items():
    print(f"  • {name}: {model_id}")


In [ ]:
# ============================================================
# 🚀 CELL 4 — MODEL LOADER
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
        
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        
        print("📥 Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        
        print("📥 Loading model...")
        
        if DEVICE == "cuda":
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float32
            )
            model = model.to(DEVICE)
        
        model.eval()
        
        print()
        print("╔══════════════════════════════════════════════╗")
        print("║          ✅ TOPAI MODEL ONLINE              ║")
        print("╚══════════════════════════════════════════════╝")
        print("🧠 Model :", MODEL_NAME)
        print("🖥️  Device:", DEVICE)


In [ ]:
# ============================================================
# 💬 CELL 5 — INFERENCE ENGINE
# ============================================================

def generate(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
        
        return tokenizer.decode(
            generated,
            skip_special_tokens=True
        )


print("✅ TopAI inference engine ready")


In [ ]:
# ============================================================
# 🤖 CELL 6 — FIRST LOCAL RESPONSE
# ============================================================

PROMPT = "Explain artificial intelligence in three concise sentences."

print("👤 USER")
print(PROMPT)
print()
print("🤖 TOPAI LOCAL MODEL")
print("─" * 60)
print(generate(PROMPT))


# ⚡ Benchmark Your Runtime

This measures generation speed on the actual machine running your notebook.

The result depends on your CPU, GPU, RAM, model, quantization, and runtime configuration.

In [ ]:
# ============================================================
# ⚡ CELL 7 — TOKENS / SECOND BENCHMARK
# ============================================================

import time

benchmark_prompt = "Write a short explanation of why local AI inference can be useful."

messages = [{"role": "user", "content": benchmark_prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

if DEVICE == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

if DEVICE == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start
generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        
        tokens_per_second = generated_tokens / elapsed if elapsed else 0
        
        print("╔══════════════════════════════════════════════╗")
        print("║             ⚡ TOPAI BENCHMARK              ║")
        print("╚══════════════════════════════════════════════╝")
        print()
        print(f"🧠 Model       : {MODEL_NAME}")
        print(f"🖥️  Device      : {DEVICE}")
        print(f"⏱️  Time        : {elapsed:.2f}s")
        print(f"🧮 New tokens  : {generated_tokens}")
        print(f"🚀 Speed       : {tokens_per_second:.2f} tokens/sec")
        print()
        print("🏁 Benchmark complete.")


# 💬 Interactive Chat

A notebook-friendly chat interface. Adjust the controls and press **Generate**.

In [ ]:
# ============================================================
# 💬 CELL 8 — INTERACTIVE CHAT UI
# ============================================================

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

prompt_box = widgets.Textarea(
    value="",
    placeholder="Ask the local model anything...",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="120px")
)

tokens_box = widgets.IntSlider(
    value=256,
    min=32,
    max=1024,
    step=32,
    description="Tokens:"
)

temperature_box = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.5,
    step=0.1,
    description="Temp:"
)

generate_button = widgets.Button(
    description="🚀 Generate",
    button_style="success",
    icon="bolt"
)

output_box = widgets.Output()

def run_chat(_):
    with output_box:
        clear_output()
        prompt = prompt_box.value.strip()
        
        if not prompt:
            print("⚠️ Enter a prompt first.")
            return
        
        print("🤖 Generating locally...\n")
        
        try:
            answer = generate(
                prompt,
                max_new_tokens=tokens_box.value,
                temperature=temperature_box.value
            )
            display(Markdown("### 🤖 TopAI\n\n" + answer))
        except Exception as error:
            print("❌ Generation failed:", error)

generate_button.on_click(run_chat)

display(prompt_box)
display(widgets.HBox([tokens_box, temperature_box]))
display(generate_button)
display(output_box)


# ☁️ Optional Cloud API Mode

Local inference is the default. If your runtime cannot handle the model you want, you can instead connect an OpenAI-compatible provider.

This section is **optional** and is not needed for local inference.

Never commit a real API key to GitHub.

In [ ]:
# ============================================================
# ☁️ CELL 9 — OPTIONAL OPENAI-COMPATIBLE API
# ============================================================

%pip install --user -q openai

import os
from getpass import getpass
from openai import OpenAI

USE_API = False

# Example only — replace these when intentionally using an API.
API_BASE_URL = "https://example.com/v1"
API_MODEL = "example-model"

def cloud_chat(prompt):
    if not USE_API:
        raise RuntimeError("USE_API is False. Local mode is currently active.")

    api_key = getpass("API key: ")

    client = OpenAI(
        base_url=API_BASE_URL,
        api_key=api_key
    )

    response = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

print("☁️ Optional API mode loaded.")
print("🔐 API keys are requested interactively and are not stored in the notebook.")


# 🏆 TopAI Scorecard

You've now got a complete local AI playground running inside your notebook runtime.

In [ ]:
# ============================================================
# 🏆 CELL 10 — TOPAI SCORECARD
# ============================================================

print("╔════════════════════════════════════════════════════╗")
print("║                 🏆 TOPAI COMPLETE                  ║")
print("╠════════════════════════════════════════════════════╣")
print("║                                                    ║")
print("║  🧠 Local model inference       ✅                 ║")
print("║  🔐 API key required            ❌                 ║")
print("║  💳 Paid API required           ❌                 ║")
print("║  👑 Root / sudo required        ❌                 ║")
print("║  🎮 Automatic GPU detection     ✅                 ║")
print("║  🐢 CPU fallback                ✅                 ║")
print("║  🤗 Hugging Face models         ✅                 ║")
print("║  💬 Interactive chat            ✅                 ║")
print("║  🎛️  Generation controls        ✅                 ║")
print("║  ⚡ Runtime benchmark            ✅                 ║")
print("║  ☁️  Optional API mode          ✅                 ║")
print("║                                                    ║")
print("║       YOUR RUNTIME. YOUR MODEL. YOUR AI.          ║")
print("║                                                    ║")
print("╚════════════════════════════════════════════════════╝")


# 🌟 What you just built

You now have a portable AI environment that can run in multiple hosted notebook platforms.

### 🖥️ Local mode

The model weights are downloaded into the runtime and generation happens there. No inference API is required.

### ☁️ API mode

If you want a larger model than your runtime can handle, TopAI can also act as an OpenAI-compatible client.

### 🚀 Why this is useful

You can fork the repository, launch the notebook, change the model, benchmark it, and experiment without rewriting the entire application.

---

## 🔗 TopAI

**GitHub:** https://github.com/ghostdev102/TopAI

**Notebook:** `ai.ipynb`

### ⭐ If this helped you, star the repository.
